In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()
hostname = os.getenv('hostname')
port = int(os.getenv('port'))
username = os.getenv('proxy_username')
password = os.getenv('proxy_password')

os.environ["ALL_PROXY"] = f"http://{username}:{password}@{hostname}:{port}"

In [ ]:
from unsloth import FastLanguageModel
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig

In [ ]:
model_name = "Qwen/Qwen3-4B-Instruct-2507"

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    load_in_4bit=True
)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=32,  # LoRA rank (size/capacity)
    target_modules=[
        "q_proj",      # Query projection
        "k_proj",      # Key projection
        "v_proj",      # Value projection
        "o_proj",      # Attention output projection

        "gate_proj",   # Gating projection
        "up_proj",     # Expands hidden features
        "down_proj",   # Reduces hidden features
    ],
    lora_alpha=32,     # Controls LoRA scaling
    lora_dropout=0,    # No dropout during training
)

In [ ]:
def formatting_func(example):
    return {"text":tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )}

In [ ]:
dataset = load_dataset(
    "json",
    data_files="dataset.jsonl",
    split="train"
)
dataset = dataset.map(formatting_func)

In [ ]:
trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    formatting_func=formatting_func,
    
    args=SFTConfig(
        output_dir="output",
        num_train_epochs=3,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        learning_rate=2e-4,
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        dataloader_pin_memory=False
    )
)

In [ ]:
trainer.train()

In [ ]:
# loss - 1.663722
model.save_pretrained("my_model1")
tokenizer.save_pretrained("my_model1")